# 07 - The complete end-to-end link
**Objective.** Connect every block and follow *one sentence* through the entire chain, then check noiseless exactness for all six MCSs on both channels.

```
text -> Huffman -> conv. encoder (K=7, 133/171) -> puncturing -> Gray QAM -> AWGN | Rayleigh block fading
     -> ZF equaliser -> max-log LLR -> depuncturing -> soft Viterbi -> Huffman decoder -> text
```
**Assumptions.** One sentence = one frame; the receiver knows the frame's number of information bits (side information, needed for trellis termination); no interleaver (not part of the specified chain); perfect CSI; each frame uses whole RBs of 84 symbols.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))          # project root = import root
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
FIG, TAB = ROOT / "results/figures", ROOT / "results/tables"
print("project root:", ROOT)

project root: /home/amardeep/Downloads/Project_Checkpoint_06/Exercise1


In [2]:
from task1_conventional.dataset import load_split
from task1_conventional.huffman import HuffmanCodec
from task1_conventional.communication_system import MCS_TABLE, ConventionalLink, Frames
from task1_conventional import convolutional as cv
from task1_conventional.metrics import bler
train, test = load_split("train", ROOT / "data"), load_split("test", ROOT / "data")
codec = HuffmanCodec.from_corpus(train)

## 1. One sentence, stage by stage (MCS4: 16-QAM 3/4, AWGN, 14 dB)

In [3]:
from task1_conventional.modulation import Modem
from task1_conventional import channel as ch
from task1_conventional.equalizer import zf_equalize
from task1_conventional.demapper import llr_maxlog
from task1_conventional.viterbi import viterbi_single
mcs = MCS_TABLE[3]; modem = Modem(mcs.modulation); rng = np.random.default_rng(7)
s = test[3]; print("1 text          :", s)
info = codec.encode(s); print(f"2 Huffman       : {len(info)} bits")
mother = cv.encode_mother(info); print(f"3 conv. encoder : {len(mother)} bits = 2*({len(info)}+6) (rate 1/2 incl. 6 tail bits)")
tx = cv.puncture(mother, mcs.rate); print(f"4 puncturing    : {len(tx)} bits (rate {mcs.rate}; info/coded = {len(info)/len(tx):.4f})")
sym = modem.modulate(tx); print(f"5 modulation    : {len(sym)} {mcs.modulation} symbols ({len(tx)} bits, {(-len(tx)) % modem.bps} padding bit(s))")
y, n0 = ch.awgn(sym, 14.0, rng); print(f"6 channel       : AWGN, Es/N0 = 14 dB, N0 = {n0:.4f}")
xh, gain = zf_equalize(y, np.ones_like(y)); llr = llr_maxlog(xh, n0 / gain, modem)[:len(tx)]
print(f"7 ZF + max-log  : {len(llr)} LLRs, raw channel bit errors = {np.sum((llr < 0) != (tx == 1))}")
dep = cv.depuncture(llr, mcs.rate, len(mother)); print(f"8 depuncturing  : {len(dep)} soft values ({np.sum(dep == 0)} erasures)")
dec = viterbi_single(dep, len(info)); print(f"9 Viterbi       : info bit errors = {np.sum(dec != info)}")
print("10 Huffman dec. :", codec.decode(dec)); print("   identical    :", codec.decode(dec) == s)

1 text          : "Yes, your honor", replied Bellows.
2 Huffman       : 175 bits
3 conv. encoder : 362 bits = 2*(175+6) (rate 1/2 incl. 6 tail bits)
4 puncturing    : 242 bits (rate 3/4; info/coded = 0.7231)
5 modulation    : 61 16QAM symbols (242 bits, 2 padding bit(s))
6 channel       : AWGN, Es/N0 = 14 dB, N0 = 0.0398
7 ZF + max-log  : 242 LLRs, raw channel bit errors = 1
8 depuncturing  : 362 soft values (120 erasures)
9 Viterbi       : info bit errors = 0
10 Huffman dec. : "Yes, your honor", replied Bellows.
   identical    : True


## 2. Noiseless exactness: all MCS x both channels

In [4]:
fr = Frames.build(test[:300], codec); rows = []
for m in MCS_TABLE:
    for chn in ("awgn", "rayleigh"):
        res, txt = ConventionalLink(codec, m, chn).run(fr, None, np.random.default_rng(0))
        rows.append(dict(MCS=m.name, channel=chn, info_bit_errors=int(res.bit_errors.sum()), BLER=bler(fr.sentences, txt)))
t = pd.DataFrame(rows); display(t); assert (t.info_bit_errors == 0).all() and (t.BLER == 0).all()
print("All noiseless transmissions reproduce the text exactly.")

,MCS,channel,info_bit_errors,BLER
0,MCS1: QPSK 1/2,awgn,0,0.0
1,MCS1: QPSK 1/2,rayleigh,0,0.0
2,MCS2: QPSK 3/4,awgn,0,0.0
3,MCS2: QPSK 3/4,rayleigh,0,0.0
4,MCS3: 16QAM 1/2,awgn,0,0.0
5,MCS3: 16QAM 1/2,rayleigh,0,0.0
6,MCS4: 16QAM 3/4,awgn,0,0.0
7,MCS4: 16QAM 3/4,rayleigh,0,0.0
8,MCS5: 64QAM 2/3,awgn,0,0.0
9,MCS5: 64QAM 2/3,rayleigh,0,0.0


All noiseless transmissions reproduce the text exactly.


## 3. How a sentence degrades with SNR (MCS4, AWGN) - first look at the cliff

In [5]:
link = ConventionalLink(codec, MCS_TABLE[3], "awgn"); one = Frames.build([test[3]], codec)
for snr in (6, 9, 10, 11, 12, 14):
    res, txt = link.run(one, float(snr), np.random.default_rng(100 + snr))
    print(f"{snr:3d} dB | info bit errors {int(res.bit_errors[0]):3d} | {txt[0]}")

  6 dB | info bit errors  68 | "hemats   our honee ihyd  o.rl h n .,dieo
  9 dB | info bit errors   0 | "Yes, your honor", replied Bellows.
 10 dB | info bit errors   0 | "Yes, your honor", replied Bellows.
 11 dB | info bit errors   0 | "Yes, your honor", replied Bellows.
 12 dB | info bit errors   0 | "Yes, your honor", replied Bellows.
 14 dB | info bit errors   0 | "Yes, your honor", replied Bellows.


## 4. Throughput of the simulator (CPU)

In [6]:
import time
fr = Frames.build(test, codec); link = ConventionalLink(codec, MCS_TABLE[2], "rayleigh"); link.run(fr, 20.0, np.random.default_rng(0))
t0 = time.time(); res, txt = link.run(fr, 20.0, np.random.default_rng(1)); dt = time.time() - t0
print(f"{len(test)} sentences, MCS3 / Rayleigh: {dt:.2f} s  ({len(test) / dt:.0f} sentences/s, vectorised batch Viterbi)")

2000 sentences, MCS3 / Rayleigh: 0.78 s  (2577 sentences/s, vectorised batch Viterbi)
